In [2]:
import torch
import random

from transformers import AutoTokenizer

#加载tokenizer
tokenizer = AutoTokenizer.from_pretrained(r'C:\Users\ssw\Desktop\CS_Code\Machine_learn\ner\model\bert-base-chinese')

tokenizer

OSError: [WinError 126] 找不到指定的模块。 Error loading "C:\Users\ssw\.conda\envs\ner\lib\site-packages\torch\lib\caffe2_nvrtc.dll" or one of its dependencies.

In [2]:
from datasets import load_dataset

#加载数据集
dataset = load_dataset(path='lansinuote/ChnSentiCorp')

#过滤句子长度
f = lambda x: len(x['text']) >= 40
dataset = dataset.filter(f)

#移除多余的字段
dataset = dataset.remove_columns(['label'])

dataset, dataset['train'][0]

Filter: 100%|██████████| 1200/1200 [00:00<00:00, 171441.00 examples/s]


(DatasetDict({
     train: Dataset({
         features: ['text'],
         num_rows: 8130
     })
     validation: Dataset({
         features: ['text'],
         num_rows: 1032
     })
     test: Dataset({
         features: ['text'],
         num_rows: 1011
     })
 }),
 {'text': '选择珠江花园的原因就是方便，有电动扶梯直接到达海边，周围餐馆、食廊、商场、超市、摊位一应俱全。酒店装修一般，但还算整洁。 泳池在大堂的屋顶，因此很小，不过女儿倒是喜欢。 包的早餐是西式的，还算丰富。 服务吗，一般'})

In [4]:
#定义数据集遍历工具
def collate_fn(data):
    b = len(data)
    text = [i['text'] for i in data]

    #生成前后两段话分别的索引
    s1 = list(range(b))
    s2 = list(range(b))
    random.shuffle(s2)

    #根据索引生成label,表明两句话是否是前后相连的关系
    label = [s1[i] == s2[i] for i in range(b)]

    #取出具体的文字
    s1 = [text[i][0:20] for i in s1]
    s2 = [text[i][20:40] for i in s2]

    #句子对编码
    data = tokenizer(s1,
                     s2,
                     padding=True,
                     truncation=True,
                     max_length=50,
                     return_tensors='pt')

    #设置label
    data['label'] = torch.LongTensor(label)

    return data


loader = torch.utils.data.DataLoader(dataset['train'],
                                     batch_size=4,
                                     shuffle=True,
                                     drop_last=True,
                                     collate_fn=collate_fn)

data = next(iter(loader))

for k, v in data.items():
    print(k, v.shape)

len(loader)# 可以看到下面的K-V键值对

input_ids torch.Size([4, 43])
token_type_ids torch.Size([4, 43])
attention_mask torch.Size([4, 43])
label torch.Size([4])


2032

In [8]:
#查看数据样例, 看[SEP]前后的句子 是否有关联(相连)
for input_ids, label in zip(data['input_ids'], data['label']):
    print(tokenizer.decode(input_ids))
    print(label)
    print('================')

[CLS] 1 ， 东 京 发 货 没 有 想 象 的 那 么 快 ， 等 了 7 天 才 [SEP] 间 比 一 般 笔 记 本 长 ， 但 比 广 告 写 的 时 间 要 短 ， [SEP]
tensor(0)
[CLS] 中 通 太 垃 圾 ， 到 了 不 通 知 ， 很 多 地 方 都 不 送 到 [SEP] 她 不 说 人 家 也 知 道, 说 了 也 白 说, 一 点 实 用 性 [SEP]
tensor(0)
[CLS] 体 积 小 ， 携 带 方 便 ， 发 热 量 不 大 ， 电 池 待 机 时 [SEP] ， 得 自 己 去 取 ， 京 东 也 太 那 个 了 ， 大 件 也 不 用 [SEP]
tensor(0)
[CLS] 全 是 一 些 虚 的 东 西, 看 过 和 没 看 一 样 好 多 东 西 [SEP] 等 到 2 ， 这 款 机 子 是 vista 克 星 ， 试 过 [SEP] [PAD] [PAD] [PAD] [PAD] [PAD]
tensor(0)


In [9]:
#定义模型
class Model(torch.nn.Module):

    def __init__(self):
        super().__init__()

        #加载预训练模型
        from transformers import AutoModel
        self.pretrained = AutoModel.from_pretrained(
            'google-bert/bert-base-chinese')

        self.fc = torch.nn.Linear(in_features=768, out_features=2)

    def forward(self, input_ids, attention_mask, token_type_ids, label=None):
        #使用预训练模型抽取数据特征
        with torch.no_grad():
            last_hidden_state = self.pretrained(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids).last_hidden_state

        #只取第0个词的特征做分类,这和bert模型的训练方式有关,此处不展开
        last_hidden_state = last_hidden_state[:, 0]

        #对抽取的特征只取第一个字的结果做分类即可
        out = self.fc(last_hidden_state).softmax(dim=1)

        #计算loss
        loss = None
        if label is not None:
            loss = torch.nn.functional.cross_entropy(out, label)

        return loss, out


model = Model()

model(**data)

C:\Users\ssw\.conda\envs\ner\lib\site-packages\huggingface_hub\file_download.py:157: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ssw\.cache\huggingface\hub\models--google-bert--bert-base-chinese. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


(tensor(0.5776, grad_fn=<NllLossBackward0>),
 tensor([[0.6436, 0.3564],
         [0.5370, 0.4630],
         [0.6256, 0.3744],
         [0.6936, 0.3064]], grad_fn=<SoftmaxBackward0>))

In [10]:
#执行训练
def train():
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    for i, data in enumerate(loader):
        loss, out = model(**data)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if i % 10 == 0:
            out = out.argmax(dim=1)
            acc = (out == data.label).sum().item() / len(data.label)
            print(i, len(loader), loss.item(), acc)

        if i == 300:
            break


train()

0 2032 0.7308163642883301 0.25
10 2032 0.5332834124565125 1.0
20 2032 0.4614488482475281 1.0
30 2032 0.440235435962677 1.0
40 2032 0.60807865858078 0.75
50 2032 0.3683162033557892 1.0
60 2032 0.4283789396286011 1.0
70 2032 0.5085047483444214 0.75
80 2032 0.574759304523468 0.5
90 2032 0.8352731466293335 0.0
100 2032 0.3500328063964844 1.0
110 2032 0.6652148962020874 0.75
120 2032 0.5056728720664978 1.0
130 2032 0.4861193299293518 0.75
140 2032 0.35024479031562805 1.0
150 2032 0.5708418488502502 0.75
160 2032 0.6641563177108765 0.75
170 2032 0.3675321936607361 1.0
180 2032 0.34973981976509094 1.0
190 2032 0.5473788976669312 0.75
200 2032 0.5245588421821594 0.5
210 2032 0.5445443391799927 0.75
220 2032 0.544475793838501 0.5
230 2032 0.3264249563217163 1.0
240 2032 0.3337784707546234 1.0
250 2032 0.4248276352882385 1.0
260 2032 0.3305405080318451 1.0
270 2032 0.3446155786514282 1.0
280 2032 0.3941941559314728 1.0
290 2032 0.38391438126564026 1.0
300 2032 0.5931883454322815 0.75


In [12]:
#执行测试
def test():
    loader_test = torch.utils.data.DataLoader(dataset['test'],
                                              batch_size=4,
                                              shuffle=True,
                                              drop_last=True,
                                              collate_fn=collate_fn)

    correct = 0
    total = 0
    for i, data in enumerate(loader_test):
        with torch.no_grad():
            _, out = model(**data)

        out = out.argmax(dim=1)
        correct += (out == data.label).sum().item()
        total += len(data.label)

        print(i, len(loader_test), correct / total)

        if i == 5:
            break

    return correct / total


test()

0 252 1.0
1 252 1.0
2 252 1.0
3 252 0.9375
4 252 0.95
5 252 0.9166666666666666


0.9166666666666666